# Treinando e avaliando modelos não supervisionados

No aprendizado **não supervisionado**, o algoritmo procura estruturas nos dados sem receber uma variável-alvo com as respostas corretas. Uma tarefa comum é o **agrupamento** (*clustering*): organizar observações semelhantes em grupos.

Neste notebook, vamos utilizar o **K-Means** para agrupar perfis de usuários e explorar métricas que ajudam a avaliar a qualidade dos agrupamentos.

## Preparando os dados

Vamos carregar o arquivo `insta_dataset.csv` e remover a coluna `User ID`. Em geral, um identificador serve para distinguir registros, mas não representa necessariamente uma característica útil para medir semelhança entre usuários.

In [ ]:
import pandas as pd
import seaborn as snb
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score
)

df = pd.read_csv("insta_dataset.csv")
df = df.drop(columns="User ID")

df.head()

Antes de treinar, vale conferir o formato dos dados e se existem valores ausentes. O K-Means espera características numéricas e não aceita valores ausentes diretamente.

In [ ]:
print("Formato da base:", df.shape)
display(df.head())
display(df.isna().sum())
display(df.dtypes)

## Treinando o K-Means

O K-Means procura dividir as observações em `n_clusters` grupos. A cada iteração, ele atribui cada observação ao centroide mais próximo e recalcula os centroides com base nas observações atribuídas.

O parâmetro `n_clusters=5` define previamente que queremos formar cinco grupos. Isso não significa que existam necessariamente cinco grupos naturais na base: esse é um valor escolhido para o experimento.

In [ ]:
model = KMeans(n_clusters=5, random_state=42, n_init=10)
clusters = model.fit_predict(df)

new_df = df.copy()
new_df["cluster"] = clusters

new_df.head()

O método `fit_predict` ajusta o modelo aos dados e retorna um rótulo de grupo para cada linha. Os números dos grupos (`0`, `1`, `2`...) são apenas identificadores: o grupo 4 não é “maior” ou “melhor” que o grupo 1.

### Quantas observações ficaram em cada grupo?

A contagem ajuda a perceber se os grupos têm tamanhos semelhantes ou se algum grupo contém poucas observações. Grupos pequenos não são necessariamente um problema, mas merecem ser investigados.

In [ ]:
new_df["cluster"].value_counts().sort_index()

## Visualizando os agrupamentos

O `pairplot` mostra gráficos de pares de variáveis e colore os pontos de acordo com o grupo atribuído pelo K-Means. Ele pode ajudar a observar separações e sobreposições entre grupos, embora a visualização possa ficar difícil quando há muitas variáveis.

In [ ]:
snb.pairplot(new_df, hue="cluster")
plt.show()

## Avaliação de agrupamentos sem rótulos verdadeiros

Em problemas supervisionados, geralmente comparamos as previsões com respostas conhecidas. Em agrupamento não supervisionado, normalmente não temos uma resposta correta para cada observação.

Por isso, usamos frequentemente **métricas internas**, calculadas a partir das características dos dados e dos grupos encontrados. Elas avaliam aspectos como proximidade dentro dos grupos e separação entre grupos.

Neste notebook veremos três métricas comuns:

- **Silhouette Score:** maior costuma ser melhor.
- **Davies–Bouldin Index:** menor costuma ser melhor.
- **Calinski–Harabasz Index:** maior costuma ser melhor.

Essas métricas observam aspectos diferentes da estrutura dos grupos. Nenhuma delas, isoladamente, comprova que os grupos são úteis para o negócio ou que representam categorias reais.

### Silhouette Score

O **Silhouette Score** compara, para cada observação, o quanto ela está próxima das observações do próprio grupo com o quanto está próxima do grupo vizinho mais adequado.

O valor médio varia de **-1 a 1**:

- próximo de **1**: observações tendem a estar bem agrupadas e separadas de outros grupos;
- próximo de **0**: há sobreposição ou observações próximas às fronteiras entre grupos;
- abaixo de **0**: algumas observações podem estar mais próximas de outro grupo do que daquele ao qual foram atribuídas.

Valores mais altos costumam indicar grupos mais compactos e separados segundo a distância utilizada.

In [ ]:
silhouette = silhouette_score(df, clusters)
print(f"Silhouette Score: {silhouette:.3f}")

### Davies–Bouldin Index

O **Davies–Bouldin Index** considera a dispersão dentro de cada grupo e a distância entre os grupos. Em termos intuitivos, ele penaliza agrupamentos que são muito espalhados internamente ou pouco separados uns dos outros.

- **Menor é melhor**: valores baixos indicam, em geral, grupos mais compactos e mais distintos.
- O índice não possui um limite superior fixo, e seu valor deve ser interpretado no contexto da comparação entre modelos e configurações.

Não compare diretamente o valor numérico dessa métrica com o Silhouette Score: elas têm escalas e interpretações diferentes.

In [ ]:
davies_bouldin = davies_bouldin_score(df, clusters)
print(f"Davies-Bouldin Index: {davies_bouldin:.3f}")

### Calinski–Harabasz Index

O **Calinski–Harabasz Index** compara a dispersão entre os grupos com a dispersão dentro dos grupos, considerando também a quantidade de grupos e observações.

- **Maior é melhor**: valores altos indicam, em geral, grupos mais separados entre si em relação à sua dispersão interna.
- O índice não tem um limite superior fixo.
- Seu valor é mais útil para comparar configurações avaliadas sobre a mesma base de dados do que para ser interpretado isoladamente.

Assim como as demais métricas internas, ele não garante que os grupos sejam úteis ou tenham significado prático.

In [ ]:
calinski_harabasz = calinski_harabasz_score(df, clusters)
print(f"Calinski-Harabasz Index: {calinski_harabasz:.3f}")

### Comparando as métricas em conjunto

Vamos reunir os três resultados em uma tabela para facilitar a leitura. Observe a direção desejável de cada métrica: **Silhouette maior**, **Davies–Bouldin menor** e **Calinski–Harabasz maior**.

In [ ]:
metricas = pd.DataFrame({
    "Métrica": [
        "Silhouette Score",
        "Davies-Bouldin Index",
        "Calinski-Harabasz Index"
    ],
    "Valor": [
        silhouette,
        davies_bouldin,
        calinski_harabasz
    ],
    "Interpretação geral": [
        "Maior tende a ser melhor",
        "Menor tende a ser melhor",
        "Maior tende a ser melhor"
    ]
})

metricas

## Comparando diferentes quantidades de grupos

O valor de `n_clusters` é uma escolha importante no K-Means. Em vez de assumir que cinco grupos é a melhor configuração, podemos treinar modelos com diferentes valores de K e comparar suas métricas internas.

O exemplo abaixo testa valores de K entre 2 e 8. O Silhouette Score e o Davies–Bouldin Index podem ser calculados para cada configuração. O Calinski–Harabasz Index também será incluído.

> Essa comparação continua sendo feita sem escalonar os dados, conforme o objetivo didático deste notebook. Como as métricas dependem das distâncias e da geometria dos dados, os resultados podem mudar bastante quando as variáveis possuem escalas diferentes.

In [ ]:
results = []

for k in range(2, 9):
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    clusters = model.fit_predict(df)

    results.append({
        "k": k,
        "silhouette": silhouette_score(df, clusters),
        "davies_bouldin": davies_bouldin_score(df, clusters),
        "calinski_harabasz": calinski_harabasz_score(df, clusters),
        "inertia": model.inertia_
    })

results_df = pd.DataFrame(results)
results_df

A tabela também inclui a **inércia** (*inertia*), que é a soma das distâncias quadráticas das observações aos centroides dos seus grupos. Ela é uma medida usada pelo K-Means para representar a compactação interna dos grupos.

A inércia tende a diminuir quando aumentamos K, porque o modelo pode formar grupos menores. Por isso, ela não deve ser usada sozinha para escolher o número de grupos; uma abordagem visual comum é procurar um “cotovelo” no gráfico, junto com outras evidências.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(results_df["k"], results_df["silhouette"], marker="o")
axes[0].set_title("Silhouette Score")
axes[0].set_xlabel("Número de grupos (K)")
axes[0].set_ylabel("Silhouette")
axes[0].grid(True)

axes[1].plot(results_df["k"], results_df["davies_bouldin"], marker="o")
axes[1].set_title("Davies-Bouldin Index")
axes[1].set_xlabel("Número de grupos (K)")
axes[1].set_ylabel("Davies-Bouldin")
axes[1].grid(True)

axes[2].plot(results_df["k"], results_df["calinski_harabasz"], marker="o")
axes[2].set_title("Calinski-Harabasz Index")
axes[2].set_xlabel("Número de grupos (K)")
axes[2].set_ylabel("Calinski-Harabasz")
axes[2].grid(True)

plt.tight_layout()
plt.show()

## Como interpretar os resultados?

Ao comparar modelos:

1. Verifique se as métricas apontam para uma configuração semelhante.
2. Observe a visualização e o tamanho dos grupos.
3. Investigue se os grupos apresentam diferenças relevantes nas variáveis.
4. Considere o objetivo da análise: grupos estatisticamente separados nem sempre são úteis na prática.

Se uma métrica favorecer um valor de K e outra favorecer um valor diferente, isso não significa necessariamente que uma delas esteja errada. Elas medem propriedades diferentes dos agrupamentos.

### E se tivermos rótulos verdadeiros?

Se a base tiver classes conhecidas por uma fonte externa, é possível comparar os grupos encontrados com essas classes usando métricas externas, como **Adjusted Rand Index (ARI)** e **Normalized Mutual Information (NMI)**. Essas métricas exigem rótulos de referência e não são necessárias para executar o agrupamento em si.

Não confunda os números de cluster produzidos pelo K-Means com classes verdadeiras: os rótulos `0`, `1`, `2` etc. são atribuídos pelo algoritmo e não têm significado semântico automático.

## Resumo das métricas

| Métrica | O que avalia, em linhas gerais | Direção desejável |
|---|---|---|
| Silhouette Score | Coesão do grupo e separação em relação a outros grupos | Maior |
| Davies–Bouldin Index | Dispersão interna e semelhança entre grupos | Menor |
| Calinski–Harabasz Index | Dispersão entre grupos em relação à dispersão interna | Maior |
| Inércia do K-Means | Soma das distâncias quadráticas aos centroides | Menor, mas tende a cair com K maior |

A avaliação não supervisionada combina métricas, visualização e conhecimento do problema. Sempre verifique se os grupos encontrados fazem sentido para o uso pretendido.